### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [14]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq
import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [15]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key= groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001A3A4C4C3A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A3A4C4EAA0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
messages=[
    SystemMessage(content= "Translate the following from English to German"),
    HumanMessage(content="Hello,How are you?")
]
result= model.invoke(messages)

In [4]:
from langchain_core.output_parsers import StrOutputParser
# StrOutputParser is used to parse the output of the model into a string format

parser= StrOutputParser()
parser.invoke(result)


'Hallo, wie geht es Ihnen?'

In [5]:
### Using LCEL- we can chain the components

chain= model|parser
chain.invoke(messages)

"Hallo, wie geht es dir?\n\n(In formal tone: \nHallo, wie geht es Ihnen?)\n\n(Note: 'Du' is informal, 'Sie' is formal. In Germany, people often use the formal 'Sie' even with friends, especially if they're slightly older or in a position of authority.)"

### Instead of giving messages every time we can directly prompt template for it.

In [10]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate
context= "Translate the following into {language}"

prompt= ChatPromptTemplate.from_messages(
    [("system", context),("user", "{text}")]
)

In [11]:
response= prompt.invoke({"language": "German", "text": "Hello,How are you?"})

In [12]:
response.to_messages()

[SystemMessage(content='Translate the following into German', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello,How are you?', additional_kwargs={}, response_metadata={})]

In [13]:
# Chaining together components with LCEL
chain= prompt|model|parser
chain.invoke({"language": "German", "text": "Hello,How are you?"})

'Hallo, wie geht es Ihnen?'